In [ ]:
from dnallm import load_config, load_model_and_tokenizer, DNAInference

In [ ]:
# Load the config file
configs = load_config("./inference_config.yaml")
configs["task"].task_type = "token"
configs["task"].num_labels = 2
configs["task"].label_names = ["NON_CODING", "CDS"]
configs["inference"].max_length = 4096

In [ ]:
# Load the model and tokenizer
model_name = "lgq12697/GENERanno-prokaryote-0.5b-cds-annotator"
# from ModelScope
model, tokenizer = load_model_and_tokenizer(model_name, task_config=configs["task"], source="modelscope")

In [ ]:
# Create inference engine
inference_engine = DNAInference(
    model=model,
    tokenizer=tokenizer,
    config=configs
)

In [ ]:

seq = (
"AGCTTTTCATTCTGACTGCAACGGGCAATATGTCTCTGTGTGGATTAAAAAAAGAGTGTC"
"TGATAGCAGCTTCTGAACTGGTTACCTGCCGTGAGTAAATTAAAATTTTATTGACTTAGG"
"TCACTAAATACTTTAACCAATATAGGCATAGCGCACAGACAGATAAAAATTACAGAGTAC"
"ACAACATCCATGAAACGCATTAGCACCACCATTACCACCACCATCACCATTACCACAGGT"
"AACGGTGCGGGCTGACGCGTACAGGAAACACAGAAAAAAGCCCGCACCTGACAGTGCGGG"
"CTTTTTTTTTCGACCAAAGGTAACGAGGTAACAACCATGCGAGTGTTGAAGTTCGGCGGT"
"ACATCAGTGGCAAATGCAGAACGTTTTCTGCGTGTTGCCGATATTCTGGAAAGCAATGCC"
"AGGCAGGGGCAGGTGGCCACCGTCCTCTCTGCCCCCGCCAAAATCACCAACCACCTGGTG"
"GCGATGATTGAAAAAACCATTAGCGGCCAGGATGCTTTACCCAATATCAGCGATGCCGAA"
"CGTATTTTTGCCGAACTTTTGACGGGACTCGCCGCCGCCCAGCCGGGGTTCCCGCTGGCG"
"CAATTGAAAACTTTCGTCGATCAGGAATTTGCCCAAATAAAACATGTCCTGCATGGCATT"
"AGTTTGTTGGGGCAGTGCCCGGATAGCATCAACGCTGCGCTGATTTGCCGTGGCGAGAAA"
"ATGTCGATCGCCATTATGGCCGGCGTATTAGAAGCGCGCGGTCACAACGTTACTGTTATC"
"GATCCGGTCGAAAAACTGCTGGCAGTGGGGCATTACCTCGAATCTACCGTCGATATTGCT"
"GAGTCCACCCGCCGTATTGCGGCAAGCCGCATTCCGGCTGATCACATGGTGCTGATGGCA"
"GGTTTCACCGCCGGTAATGAAAAAGGCGAACTGGTGGTGCTTGGACGCAACGGTTCCGAC"
"TACTCTGCTGCGGTGCTGGCTGCCTGTTTACGCGCCGATTGTTGCGAGATTTGGACGGAC"
"GTTGACGGGGTCTATACCTGCGACCCGCGTCAGGTGCCCGATGCGAGGTTGTTGAAGTCG"
"ATGTCCTACCAGGAAGCGATGGAGCTTTCCTACTTCGGCGCTAAAGTTCTTCACCCCCGC"
"ACCATTACCCCCATCGCCCAGTTCCAGATCCCTTGCCTGATTAAAAATACCGGAAATCCT"
"CAAGCACCAGGTACGCTCATTGGTGCCAGCCGTGATGAAGACGAATTACCGGTCAAGGGC"
"ATTTCCAATCTGAATAACATGGCAATGTTCAGCGTTTCTGGTCCGGGGATGAAAGGGATG"
"GTCGGCATGGCGGCGCGCGTCTTTGCAGCGATGTCACGCGCCCGTATTTCCGTGGTGCTG"
"ATTACGCAATCATCTTCCGAATACAGCATCAGTTTCTGCGTTCCACAAAGCGACTGTGTG"
"CGAGCTGAACGGGCAATGCAGGAAGAGTTCTACCTGGAACTGAAAGAAGGCTTACTGGAG"
"CCGCTGGCAGTGACGGAACGGCTGGCCATTATCTCGGTGGTAGGTGATGGTATGCGCACC"
"TTGCGTGGGATCTCGGCGAAATTCTTTGCCGCACTGGCCCGCGCCAATATCAACATTGTC"
)

In [ ]:
# Perform inference
results = inference_engine.infer_seqs([seq])

In [ ]:
for i, label in enumerate(results[0]["label"]):
    print(i, label, results[0]["scores"][i], sep=", ")

In [ ]:
import pandas as pd
from tqdm import tqdm
from dnallm.inference.plot import plot_annotations

In [ ]:
# !wget -c https://huggingface.co/datasets/GenerTeam/cds-annotation/resolve/main/bacteria_annotation_summary.parquet

In [ ]:
annotation_results = "/bacteria_annotation_summary.parquet"
data = pd.read_parquet(annotation_results, engine="pyarrow")
# select E. coli genome
seq_len = len(data.iloc[20].sequence)

In [3]:
annotations = {}
for model in data.columns[-8:]:
    annotations[model] = {"Non-Coding": set(), "CDS+": set(), "CDS-": set(), "Overlapping_CDS": set()}
    for i, l in tqdm(enumerate(data.iloc[20][model]), desc=f"Processing {model}"):
        if i < seq_len:
            if l:
                annotations[model]["CDS+"].add(i)
        else:
            i2 = i - seq_len
            if l:
                if i2 in annotations[model]["CDS+"]:
                    annotations[model]["Overlapping_CDS"].add(i2)
                    annotations[model]["CDS+"].remove(i2)
                else:
                    annotations[model]["CDS-"].add(i2)
    for i in range(seq_len):
        if i not in annotations[model]["CDS+"] and i not in annotations[model]["CDS-"]:
            if i not in annotations[model]["Overlapping_CDS"]:
                annotations[model]["Non-Coding"].add(i)

Processing label_cds_pseudo: 9283304it [00:02, 3423580.06it/s]
Processing MetaProdigal: 9283304it [00:02, 3849307.01it/s]
Processing MetaGeneMark2: 9283304it [00:02, 3778448.54it/s]
Processing Glimmer3: 9283304it [00:02, 3704093.65it/s]
Processing Prodigal: 9283304it [00:02, 3621979.72it/s]
Processing GeneLM: 9283304it [00:02, 3890068.92it/s]
Processing GeneMarkS2: 9283304it [00:02, 3859301.95it/s]
Processing GENERanno: 9283304it [00:02, 3742033.78it/s]


In [16]:
print(sorted(annotations["GENERanno"]["Overlapping_CDS"]))

[229966, 229967, 229968, 229969, 235534, 235535, 235536, 235537, 245064, 245065, 245066, 245067, 245068, 245069, 245070, 245071, 245072, 245073, 245074, 245075, 245076, 245077, 245078, 245079, 245080, 245081, 245082, 245083, 245084, 245085, 245086, 245087, 245088, 245089, 245090, 245091, 245092, 245093, 250041, 250042, 250043, 250044, 250045, 250046, 250047, 250048, 250049, 250050, 250051, 250052, 250053, 250054, 250055, 250056, 250057, 250058, 250059, 250060, 250061, 250062, 250063, 250064, 250065, 250066, 250067, 250068, 250069, 499013, 499014, 499015, 499016, 519732, 519733, 519734, 519735, 519736, 519737, 519738, 519739, 519740, 519741, 519742, 519743, 519744, 519745, 519746, 519747, 519748, 519749, 519750, 519751, 519752, 519753, 519754, 519755, 519756, 519757, 519758, 519759, 519760, 519761, 566381, 566382, 566383, 582151, 582152, 582153, 582154, 582155, 582156, 582157, 582158, 582159, 582160, 582161, 582162, 582163, 582164, 582165, 582166, 582167, 582168, 582169, 582170, 582171,

In [21]:
chart = plot_annotations(annotations, start=3400000, end=3450000, width=800)
chart

alt.Chart(...)